# Fine-tuning Paraphrase Multilingual MPNet for Burmese Headline Generation

This notebook fine-tunes the `paraphrase-multilingual-mpnet-base-v2` model for generating Burmese news headlines from article text.

**Note**: Since MPNet is an encoder-only model, we'll adapt it for generation by either:
1. Using it as an encoder in an encoder-decoder architecture
2. Fine-tuning a separate seq2seq model with MPNet embeddings

We'll use approach 2 with mT5 for better multilingual support.

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q transformers datasets sentence-transformers accelerate evaluate rouge-score sacrebleu

In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from sentence_transformers import SentenceTransformer
import evaluate
from google.colab import drive

# Mount Google Drive (optional - for saving models)
# drive.mount('/content/drive')

In [ ]:
# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Prepare Burmese Dataset

You'll need a dataset with Burmese articles and headlines. Here are some options:
- Use existing Burmese news datasets
- Load from CSV/JSON files
- Scrape Burmese news websites (with permission)

Expected format: `{'article': 'article text...', 'headline': 'headline text...'}`

In [ ]:
# Example: Create sample Burmese data (replace with your actual dataset)
sample_data = {
    'article': [
        "မြန်မာနိုင်ငံ၏ စီးပွားရေးသည် ယခုနှစ်တွင် တိုးတက်မှုရှိလာနေပါသည်။ နိုင်ငံတကာရင်းနှီးမှုများ ဝင်ရောက်လာခြင်းကြောင့် အလုပ်အကိုင်အခွင့်အလမ်းများ များပြားလာပါသည်။",
        "ရန်ကုန်မြို့တွင် နည်းပညာကုမ္ပဏီအသစ်များ စတင်ဖွင့်လှစ်လျက်ရှိပါသည်။ ၎င်းတို့သည် လူငယ်များအတွက် အလုပ်အကိုင်များစွာ ဖန်တီးပေးနေပါသည်။",
        "ပညာရေးဝန်ကြီးဌာနက အခြေခံပညာသင်ကြားမှုစနစ် တိုးတက်မွမ်းမံရေး အစီအစဉ်များကို ကြေညာခဲ့ပါသည်။"
    ],
    'headline': [
        "မြန်မာ့စီးပွားရေး တိုးတက်မှုရှိ",
        "ရန်કုန်တွင် နည်းပညာကုမ္ပဏီများ ဖွင့်လှစ်",
        "ပညာရေးစနစ် မွမ်းမံမည်"
    ]
}

# Convert to DataFrame
df = pd.DataFrame(sample_data)
print(f"Dataset size: {len(df)} samples")
df.head()

In [ ]:
# OPTION: Load your own data from CSV
# df = pd.read_csv('burmese_news.csv')
# Make sure it has 'article' and 'headline' columns

# OPTION: Load from Google Drive
# df = pd.read_csv('/content/drive/MyDrive/burmese_headlines.csv')

# OPTION: Load from Hugging Face datasets
# from datasets import load_dataset
# dataset = load_dataset('your-burmese-dataset-name')

In [ ]:
# Split data into train/validation/test
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Convert to Hugging Face Dataset
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(val_df),
    'test': Dataset.from_pandas(test_df)
})

print(dataset)

## 3. Load Model and Tokenizer

We'll use mT5 (multilingual T5) which supports Burmese well for seq2seq tasks.

In [ ]:
# Use mT5 for better multilingual support including Burmese
model_name = "google/mt5-small"  # Options: mt5-small, mt5-base, mt5-large

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print(f"Model loaded: {model_name}")
print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
# Optional: Load MPNet for semantic similarity evaluation
mpnet_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
print("MPNet model loaded for evaluation")

## 4. Preprocess Data

In [ ]:
# Set max lengths
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    """Tokenize articles and headlines"""
    # Add prefix for better task understanding
    inputs = ["summarize: " + doc for doc in examples["article"]]
    
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )
    
    # Tokenize targets (headlines)
    labels = tokenizer(
        examples["headline"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply preprocessing
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("Tokenization complete!")
print(tokenized_dataset)

## 5. Training Setup

In [ ]:
# Load evaluation metrics
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    """Compute ROUGE scores for evaluation"""
    predictions, labels = eval_pred
    
    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Replace -100 in labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Compute ROUGE scores
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    
    # Extract key metrics
    result = {key: value * 100 for key, value in result.items()}
    
    return {k: round(v, 4) for k, v in result.items()}

In [ ]:
# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./burmese-headline-generation",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=10,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
    logging_dir="./logs",
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rouge1",
    push_to_hub=False,
    report_to="none"  # Change to "tensorboard" or "wandb" if you want logging
)

print("Training arguments configured")

In [ ]:
# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

In [ ]:
# Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer initialized and ready!")

## 6. Train the Model

In [ ]:
# Start training
print("Starting training...")
trainer.train()
print("Training complete!")

## 7. Evaluate the Model

In [ ]:
# Evaluate on test set
eval_results = trainer.evaluate(tokenized_dataset["test"])
print("\nEvaluation Results:")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

## 8. Test Headline Generation

In [ ]:
def generate_headline(article_text, max_length=128, num_beams=4):
    """Generate headline from article text"""
    # Prepare input
    input_text = "summarize: " + article_text
    inputs = tokenizer(
        input_text,
        max_length=max_input_length,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # Generate
    model.to(device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_beams=num_beams,
        early_stopping=True,
        no_repeat_ngram_size=3,
        length_penalty=1.0
    )
    
    # Decode
    headline = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return headline

In [ ]:
# Test with examples from test set
test_examples = dataset["test"].select(range(min(3, len(dataset["test"]))))

print("=" * 80)
print("HEADLINE GENERATION EXAMPLES")
print("=" * 80)

for idx, example in enumerate(test_examples):
    article = example["article"]
    true_headline = example["headline"]
    generated_headline = generate_headline(article)
    
    print(f"\nExample {idx + 1}:")
    print("-" * 80)
    print(f"Article: {article[:200]}..." if len(article) > 200 else f"Article: {article}")
    print(f"\nTrue Headline: {true_headline}")
    print(f"Generated: {generated_headline}")
    print("-" * 80)

In [ ]:
# Interactive testing - Try your own Burmese text
custom_article = """
ရန်ကုန်မြို့ရှိ ဈေးကွက်များတွင် ဒီဇင်ဘာလအတွင်း 
စားသောက်ကုန်ဈေးနှုန်းများ သိသိသာသာ မြင့်တက်လာခဲ့ပါသည်။
"""

print("Custom Article:")
print(custom_article)
print("\nGenerated Headline:")
print(generate_headline(custom_article.strip()))

## 9. Save the Model

In [ ]:
# Save model locally
output_dir = "./burmese-headline-model-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

# Optional: Save to Google Drive
# !cp -r {output_dir} /content/drive/MyDrive/
# print("Model copied to Google Drive")

## 10. Load Saved Model (for future use)

In [ ]:
# Load the saved model
# loaded_model = AutoModelForSeq2SeqLM.from_pretrained(output_dir)
# loaded_tokenizer = AutoTokenizer.from_pretrained(output_dir)
# print("Model loaded successfully!")

## 11. Export for Production (Optional)

In [ ]:
# Optional: Convert to ONNX for faster inference
# !pip install -q optimum[exporters]

# from optimum.onnxruntime import ORTModelForSeq2SeqLM

# ort_model = ORTModelForSeq2SeqLM.from_pretrained(
#     output_dir,
#     export=True
# )
# ort_model.save_pretrained("./burmese-headline-onnx")
# print("ONNX model exported")

## Notes and Tips

### Improving Performance:
1. **More Data**: Collect more Burmese news articles with headlines (1000+ examples recommended)
2. **Larger Model**: Try `google/mt5-base` or `google/mt5-large` for better quality
3. **Data Augmentation**: Back-translation or paraphrasing of existing data
4. **Hyperparameter Tuning**: Adjust learning rate, batch size, num_beams
5. **Preprocessing**: Clean and normalize Burmese text properly

### Model Options:
- `google/mt5-small`: Fast, good for prototyping (~300M params)
- `google/mt5-base`: Better quality (~580M params)
- `google/mt5-large`: Best quality (~1.2B params, requires more GPU)

### Dataset Sources:
- Burmese news websites
- Myanmar Wikipedia articles
- Public Burmese NLP datasets

### GPU Memory Tips:
- Reduce `per_device_train_batch_size` if OOM error
- Use gradient accumulation: `gradient_accumulation_steps=2`
- Enable `fp16=True` for mixed precision training
